|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>Observability<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: six numbers, and what each one is asking you to fix<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Compute the six numbers from a recorded run, then use them to answer a
question a single latency number cannot.

All arithmetic on arrays. The interesting part is Exercise 3.

In [ ]:
### run this cell: one server run, already recorded
NUM_REQUESTS = 4000
arrivals = np.cumsum(rng.exponential(1/25, size=NUM_REQUESTS))
queue_wait = rng.exponential(0.8, size=NUM_REQUESTS)          # the wait before the first step
output_lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=NUM_REQUESTS).astype(int) + 1
token_interval = rng.lognormal(mean=np.log(0.012), sigma=0.4, size=NUM_REQUESTS)  # s for each token
prefill_s = 0.04 * rng.lognormal(mean=np.log(400), sigma=0.6, size=NUM_REQUESTS)/1000
first_token = arrivals + queue_wait + prefill_s
finish = first_token + output_lengths*token_interval
print(f'{NUM_REQUESTS} requests over {finish.max():.0f} seconds')

# Exercise 1: the metrics

TTFT, TPOT, end-to-end latency, throughput, queue wait. Percentiles, not
means.

In [ ]:
# TTFT is from the arrival to the FIRST token. So it holds the queue wait AND
# the prefill. TPOT is the time for each token after that. Latency is all of it.
ttft = first_token - arrivals
tpot = token_interval
latency = finish - arrivals
total_tokens = output_lengths.sum()
wall = finish.max()

def percentile(values, rank):
  return float(np.percentile(values, rank))

print(f"{'metric':<12} {'p50':>9} {'p90':>9} {'p99':>9}")
for name, values in (('TTFT (s)', ttft), ('TPOT (ms)', tpot*1000),
                     ('latency (s)', latency)):
  print(f'{name:<12} {percentile(values,50):>9.3f} {percentile(values,90):>9.3f} '
        f'{percentile(values,99):>9.3f}')
print(f'\nthroughput  {total_tokens/wall:8.0f} tokens/s, {NUM_REQUESTS/wall:.1f} requests/s')
print(f'queue wait  {np.median(queue_wait):8.3f} s median')

# Exercise 2: goodput against three promises

A request is good only if it met **both** its TTFT and TPOT targets. Count
those, and compare with raw throughput.

In [ ]:
def goodput(ttft_sla, tpot_sla):
  """-> (the fraction of requests that met BOTH promises, the rate of them)."""
  met = (ttft <= ttft_sla) & (tpot <= tpot_sla)
  return met.mean(), met.sum()/wall

print(f"{'TTFT SLA':>9} {'TPOT SLA':>10} {'met':>7} {'goodput req/s':>14}")
for ttft_sla, tpot_sla in ((1.0, 0.020), (2.0, 0.030), (5.0, 0.050)):
  fraction_met, goodput_rate = goodput(ttft_sla, tpot_sla)
  print(f'{ttft_sla:>8.1f}s {tpot_sla*1000:>9.0f}ms {100*fraction_met:>6.1f}% '
        f'{goodput_rate:>14.1f}')
print(f'\nraw throughput {NUM_REQUESTS/wall:.1f} requests/s, and it does not change')

# Exercise 3: which promise did they break?

This is the exercise. A request can miss because it waited to start, or
because it generated slowly, and those are different bugs with different
fixes.

In [ ]:
# One 'p99 latency' number cannot tell you what to repair. Divide the misses
# by the promise that each one broke.
TTFT_SLA, TPOT_SLA = 1.0, 0.020
slow_start = (ttft > TTFT_SLA) & (tpot <= TPOT_SLA)
slow_tokens = (ttft <= TTFT_SLA) & (tpot > TPOT_SLA)
both = (ttft > TTFT_SLA) & (tpot > TPOT_SLA)
print(f'missed on TTFT only:  {100*slow_start.mean():5.1f}%  -> queueing or prefill')
print(f'missed on TPOT only:  {100*slow_tokens.mean():5.1f}%  -> the step is too slow')
print(f'missed on both:       {100*both.mean():5.1f}%')
late = ttft > TTFT_SLA
queue_share = np.median(queue_wait[late]) / np.median(ttft[late])
print(f'\nof the TTFT misses, the queue wait is {100*queue_share:.0f}% of the median TTFT')

# Exercise 4: look at the shapes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].hist(ttft, bins=60)
axes[0].set(xlabel='TTFT (s)', ylabel='requests', title='Time to first token')
axes[0].axvline(TTFT_SLA, color='r', ls='--')
axes[1].hist(tpot*1000, bins=60)
axes[1].set(xlabel='TPOT (ms)', title='For each output token')
axes[1].axvline(TPOT_SLA*1000, color='r', ls='--')
axes[2].hist(latency, bins=60)
axes[2].set(xlabel='Total latency (s)', title='End to end')
for axis in axes:
  axis.grid(alpha=.3)
plt.tight_layout()
plt.show()
print(f'TTFT    mean {ttft.mean():.3f}s  median {np.median(ttft):.3f}s  '
      f'p99 {np.percentile(ttft,99):.3f}s')
print(f'the mean is {ttft.mean()/np.median(ttft):.1f}x the median: a long right tail')

### What you can now answer that a latency number cannot

Exercise 3 earns its place. "p99 latency is 12 seconds" is a fact with no
action. Divide the misses, and you get an action:

- **mostly TTFT misses** means that requests wait. So correct the admission
  and the concurrency. You probably also need prefix caching for the prefill
  half. See stages 09, 10 and 11.
- **mostly TPOT misses** means that the step itself is slow. So correct the
  kernel, the batch size, or the CUDA graphs. See stages 08 and 12.
- **both** means that you are above capacity. No amount of tuning helps. Add
  hardware, or refuse some of the load.

These are three completely different projects. The single number that you
started with pointed at none of them.

### And the mean is a trap

Look at the ratio in Exercise 4. The mean TTFT is much above the median,
because the distribution has a long right tail. Every queueing system has one.
A dashboard that reports an average looks healthy through an outage that half
of your users see.

Report percentiles. And know which population they cover. That is the trap in
the chunked-prefill challenge in Part 4.

    ./vc guide 16